# ISM 6564 — Assignment 3: build the ticket router

**Northwind Outfitters, support operations.** 3,000 routed tickets in, one
working router out: a model, a triage threshold, a routing file for this
morning's 400-ticket batch, and a memo the operations lead can plan around.
The full brief, the rubric, and the submission list are on the
**assignment page in Canvas** — read it first. This notebook needs nothing
else: the data loads itself from the public course data repository.

How this notebook works:

- Cells marked **`GIVEN`** are infrastructure. Run them; do not modify them.
- Cells marked **`TODO`** are yours. Each defines its outputs as `None` first —
  replace the `None`s with real work.
- The cell after each TODO **checks** your work and prints a clear message
  while anything is missing, so the notebook always runs top to bottom.
  Do not delete the check cells; they are how the work is marked.

In [41]:
# GIVEN — do not modify. Environment guard: versions and determinism.
import sys
import numpy as np
import pandas as pd
import sklearn
import scipy
import xgboost

SEED = 42
np.random.seed(SEED)

print(f"python  {sys.version.split()[0]}")
print(f"numpy   {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__}")

python  3.13.15
numpy   2.1.3 | pandas 2.2.3 | sklearn 1.6.1 | xgboost 3.4.1


---

## The inbox

Two files. `tickets.csv` is what the dispatchers routed before they fell
behind — that is your labelled history. `incoming_batch.csv` is this morning's
unrouted batch — no `queue` column, because routing it is the job.

The text a router sees is `subject` + `body`, concatenated. `channel` rides
along as metadata; Task 5's menu includes testing whether it earns a place as
a feature.

In [42]:
# GIVEN — do not modify. Data loads straight from the course data repository,
# so this notebook runs anywhere — a clone, a bare download, or Colab.
BASE_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-03-text-classification/data/")

tickets = pd.read_csv(BASE_URL + "tickets.csv").fillna("")
batch = pd.read_csv(BASE_URL + "incoming_batch.csv").fillna("")

tickets["text"] = (tickets["subject"] + " " + tickets["body"]).str.strip()
batch["text"] = (batch["subject"] + " " + batch["body"]).str.strip()

QUEUES = sorted(tickets["queue"].unique())

assert len(tickets) == 3000 and len(batch) == 400
assert "queue" not in batch.columns
print(f"{len(tickets)} labelled tickets | {len(batch)} to route | queues: {QUEUES}")

3000 labelled tickets | 400 to route | queues: ['account-access', 'billing', 'product-question', 'returns-warranty', 'shipping']


### The results registry

Every model run in this notebook goes through `record(...)`, so Task 4 ends
with one sortable table instead of numbers scattered across forty cells.
`evaluate(...)` computes the three scores this assignment reports everywhere:
**accuracy** (what Maya's dashboard will show), **macro-F1** (every queue
counts equally — the small queue cannot hide), and **weighted-F1** (queues
count by size).

In [43]:
# GIVEN — do not modify. Scoring and the registry.
import time
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

RESULTS = []


def evaluate(y_true, y_pred):
    """(accuracy, macro_f1, weighted_f1) for one set of predictions."""
    return (accuracy_score(y_true, y_pred),
            f1_score(y_true, y_pred, average="macro"),
            f1_score(y_true, y_pred, average="weighted"))


def record(name, y_true, y_pred, seconds=None, note=""):
    acc, mf1, wf1 = evaluate(y_true, y_pred)
    RESULTS.append({"model": name, "accuracy": round(acc, 4),
                    "macro_f1": round(mf1, 4), "weighted_f1": round(wf1, 4),
                    "fit_seconds": None if seconds is None else round(seconds, 1),
                    "note": note})
    print(f"{name:34} acc {acc:.4f}  macroF1 {mf1:.4f}  weightedF1 {wf1:.4f}")
    return acc, mf1, wf1


def results_table():
    return pd.DataFrame(RESULTS).sort_values("macro_f1", ascending=False).reset_index(drop=True)


def show_confusion(y_true, y_pred, labels=None):
    labels = labels or QUEUES
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return pd.DataFrame(cm, index=[f"true:{l}" for l in labels],
                        columns=[l[:10] for l in labels])

---

## Task 1 — Meet the inbox

Numbers before models. Build `task1_table`: one row per queue with the ticket
`count` and `share` (fraction of the corpus), sorted by count. Then
`length_by_channel`: median and mean **word count** of `text` per channel.
Then `shortest`: the 8 shortest tickets by character length of `text`, showing
`ticket_id`, `channel`, `text`, and `queue` — and actually read them.

In [44]:
# TODO (1): the three views of the inbox.

# 1. task1_table — one row per queue, count and share, sorted by count desc
task1_table = (
    tickets["queue"]
    .value_counts()
    .reset_index(name="count")
    .assign(share=lambda x: x["count"] / len(tickets))
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

# 2. length_by_channel — median/mean word count of text per channel
length_by_channel = (
    tickets.assign(word_count=tickets["text"].str.split().str.len())
    .groupby("channel")["word_count"]
    .agg(median_words="median", mean_words="mean")
    .reset_index()
)

# 3. shortest — 8 shortest tickets by character length of text
pd.set_option("display.max_colwidth", None)
shortest = tickets.assign(char_len=tickets["text"].str.len()).nsmallest(
    8, "char_len"
)[["ticket_id", "channel", "text", "queue"]]


In [45]:
# CHECK (1) — leave as is.
if task1_table is None or length_by_channel is None or shortest is None:
    print("Task 1 not complete yet: fill in task1_table, length_by_channel, and shortest above.")
else:
    assert set(map(str.lower, task1_table.columns)) >= {"queue", "count", "share"}
    assert abs(task1_table["share"].sum() - 1.0) < 1e-6, "shares must sum to 1"
    assert len(shortest) == 8
    display(task1_table); display(length_by_channel); display(shortest)

,queue,count,share
0,shipping,890,0.296667
1,returns-warranty,637,0.212333
2,billing,589,0.196333
3,product-question,531,0.177000
4,account-access,353,0.117667


,channel,median_words,mean_words
0,chat,27.0,26.324918
1,email,32.0,33.596618
2,webform,30.0,32.205431


,ticket_id,channel,text,queue
554,T-10555,webform,?????,product-question
1071,T-11072,email,?????,product-question
1355,T-11356,chat,?????,shipping
2313,T-12314,chat,?????,shipping
2466,T-12467,chat,?????,returns-warranty
337,T-10338,webform,?????!!,product-question
921,T-10922,email,?????!!,billing
2290,T-12291,webform,?????!!,billing


### Task 1 write-up

Shipping dominates at 29.7% of the corpus, roughly 1.5x the next-largest queue
(returns-warranty, 21.2%) and 2.5x the smallest (account-access, 11.8%) —
enough imbalance that a majority-class baseline already clears 30% accuracy,
which is worth remembering when judging any later model's headline number.
Chat tickets run noticeably shorter than email (median 27 words vs 32), with
webform in between (30); chat's brevity means a bag-of-words router has fewer
word co-occurrences to key off per ticket, so short chat tickets should be
among the least confident predictions downstream. The eight shortest tickets
are almost entirely punctuation noise ("?????", "?????!!") with no real words
at all — a TF-IDF vector of a genuinely short ticket like "see attached
screenshot for order NW-51234" would fare only slightly better: after the
order number (out-of-vocabulary, or at best a rare token) it is left with
"see", "attached", "screenshot", "order" as its only signal, none of which is
queue-specific on its own.

---

## Task 2 — Split like the test set is tomorrow's inbox

Dispatchers resubmit; the corpus contains **exact duplicate bodies**. A
duplicate that lands one copy in train and one in validation is a validation
row your model has memorized — your score is partly a memory test.

Three steps, in this order:

1. `n_dup` — how many tickets share a `body` with an earlier ticket
   (`tickets.duplicated("body")` counts exactly this).
2. `n_straddle` — make a **naive** 75/25 split of the *un*-deduplicated corpus
   (stratified on `queue`, `random_state=SEED`) and count validation rows
   whose `body` also appears in that split's training rows. Those are the
   memorized rows. (This split is a demonstration — throw it away afterwards.)
3. The real thing: `dedup` = tickets with duplicate bodies dropped (junk
   tickets can collide by accident too — dropping those copies is fine), then
   a stratified 75/25 split of `dedup` into `train` and `val`,
   `random_state=SEED`.

In [46]:
# TODO (2): duplicates, the straddle count, and the honest split.
from sklearn.model_selection import train_test_split

n_dup = tickets.duplicated("body").sum()
train, val = train_test_split(tickets, test_size=0.25, stratify=tickets["queue"], random_state=SEED)
n_straddle = val["body"].isin(train["body"]).sum()  # int: naive-split validation rows with a body twin in naive-split train
dedup = tickets.drop_duplicates("body").reset_index(drop=True)     # DataFrame: tickets, duplicate bodies dropped
train, val = train_test_split(dedup, test_size=0.25, stratify=dedup["queue"], random_state=SEED)


In [47]:
# GIVEN CHECK (2) — do not modify. No overlap, shares held.
if any(x is None for x in (n_dup, n_straddle, dedup, train, val)):
    print("Task 2 not complete yet: fill in n_dup, n_straddle, dedup, train, val above.")
else:
    assert n_dup > 0, "there really are duplicates - look again"
    assert len(dedup) == 3000 - n_dup
    assert len(train) + len(val) == len(dedup)
    overlap = set(train["body"]) & set(val["body"])
    assert not overlap, f"{len(overlap)} bodies appear in BOTH train and val - dedup before splitting"
    share_gap = (train["queue"].value_counts(normalize=True)
                 - val["queue"].value_counts(normalize=True)).abs().max()
    assert share_gap < 0.02, f"stratification looks broken: max share gap {share_gap:.3f}"
    print(f"duplicates: {n_dup} | naive-split memorized rows: {n_straddle}")
    print(f"train {len(train)} / val {len(val)} - no text overlap, shares within {share_gap:.1%}. Split is honest.")

duplicates: 70 | naive-split memorized rows: 22
train 2197 / val 733 - no text overlap, shares within 0.1%. Split is honest.


### Task 2 write-up

*Replace this text: 2–3 sentences. Why must deduplication happen before the
split and not after? Which direction would the memorized rows have pushed your
validation score, and why does the size of the push not change the argument?*

Deduplication must happen before the split because deduplication prevents leakages for the fit. The duplicated data would push the rows around 1 point positively. The size of the push does not change the argument because the duplicated terms are not used. However, it does change the coeficient.

---

## Task 3 — The baseline you could ship this afternoon

Two models: `DummyClassifier(strategy="most_frequent")` — the score you get by
routing everything to the biggest queue — and the Week 2 finale pipeline,
`TfidfVectorizer()` into `LogisticRegression(max_iter=2000)`. Fit on `train`
(text column), evaluate on `val`, and `record(...)` both.

Then look at *how* the baseline fails: the confusion matrix
(`show_confusion`), and `misroutes` — three actual validation tickets the
model got wrong, with `ticket_id`, true `queue`, predicted queue, and `text`.
Pick three that look *reasonable* rather than junk; you are looking for the
pattern, not the garbage.

In [48]:
# TODO (3): two baselines, the confusion matrix, three read misroutes.
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


baseline_pred = None   # np.ndarray: DummyClassifier predictions on val
logreg_pipe = None     # fitted Pipeline: TfidfVectorizer -> LogisticRegression
logreg_pred = None     # np.ndarray: its predictions on val
misroutes = None       # DataFrame: 3 misrouted val tickets (ticket_id, queue, predicted, text)

X_tr, y_tr = train["text"], train["queue"]
# your code here — remember record("majority-class", ...) and record("tfidf + logreg", ...)
dummy= DummyClassifier(strategy="most_frequent").fit(X_tr,y_tr)

baseline_pred = dummy.predict(val["text"])

tvec = TfidfVectorizer(min_df=3)
tXtr= tvec.fit_transform(X_tr)
logreg_pipe= LogisticRegression(max_iter=1000).fit(tXtr, y_tr)
logreg_pred= logreg_pipe.predict(tvec.transform(val["text"]))
misroutes= val.assign(predicted=logreg_pred).query("queue != predicted").sample(3, random_state=SEED)[["ticket_id", "queue", "predicted", "text"]]

terr = np.where(logreg_pred != val["queue"].values)[0]
for k in terr[:8]:
    row = val.iloc[k]
    print(f"[{row['queue']} -> {logreg_pred[k]}] {row['text'][:110]}")

tcm=confusion_matrix(val["queue"], logreg_pred, labels=QUEUES)

print(tcm)



[returns-warranty -> shipping] Need assistance I was charged a restocking fee of $58.00 on my return - nothing on the site mentioned that. my
[shipping -> account-access] Urgent Hi there, I paid $42.50 extra for two day shipping and it has been four days. how do i chhange the emai
[shipping -> billing] Urgent Hey, Where is my order? The Glacier 45 cooler was supposed to arrive yesterday. you charged the wrong c
[product-question -> returns-warranty] Support request Good morning, Is the Ridgeline 4P tent a true 4-season tent or more of a summer shelter. i sen
[account-access -> product-question] Need assistance I changed phones and can no longer get the two factor authentication code. can i machine wash 
[shipping -> billing] Delivery problem Hello, Can I change the delivery address on order NW-75333 before it ships. the price of the 
[product-question -> returns-warranty] Help Good morning, Do the Trailhead hiking boots run true to size - I am usually a 9.5. my exchange shipped bu
[ret

In [49]:
# CHECK (3) — leave as is.
if any(x is None for x in (baseline_pred, logreg_pipe, logreg_pred, misroutes)):
    print("Task 3 not complete yet: fill in baseline_pred, logreg_pipe, logreg_pred, misroutes above.")
else:
    assert len(logreg_pred) == len(val)
    assert len(misroutes) == 3 and (misroutes["queue"] != misroutes["predicted"]).all()
    display(show_confusion(val["queue"], logreg_pred))
    for r in misroutes.itertuples():
        print(f"\n[{r.ticket_id}] true={r.queue}  predicted={r.predicted}\n  {r.text[:300]}")

,account-ac,billing,product-qu,returns-wa,shipping
true:account-access,73,4,4,3,3
true:billing,2,135,1,3,4
true:product-question,4,3,113,6,3
true:returns-warranty,5,3,2,142,3
true:shipping,3,7,4,3,200



[T-12283] true=shipping  predicted=account-access
  Support request Hello, FedEx tracking shows my package has been sitting in Memphis for three days. i cannot log in to my account - it keeps saying my password is wrong even after i reset it. how do i change the email address on my account - i no longer have access to the old one. This is for an upco

[T-11947] true=returns-warranty  predicted=shipping
  Need assistance I was charged a restocking fee of $58.00 on my return - nothing on the site mentioned that. my refund was approved but the replacement drifter 10 kayak still has no tracking number??

[T-11275] true=shipping  predicted=returns-warranty
  Urgent Hey, If I refuse the package at delivery do I still have to pay return shipping. is the anchor bear canister still returnable if i opened the packaging. i was charged a restocking fee of $42.99 on my return - nothing on the site mentioned that. I need this sorted before the end of the month. 


### Task 3 write-up

*Replace this text: 4–6 sentences. Which queue pair confuses most, and does
that match how the queues share vocabulary? Do macro-F1 and weighted-F1
disagree — and given that `account-access` is the smallest queue, what does
the direction of the disagreement tell you? What do your three quoted
misroutes have in common? (Look closely — do any of them ask two things at
once? Keep that thought for Task 5.)*

Shipping queues are most mistaken for account access. The macro-F1 and weighted F-1 do agree. The direction of the disagreement, especially for account-access suggests that it is often mistaken due to lower occurances. The three misroute quotes have users asking two questions at once. The predicted queue categorizes based on the later question, while the true label categorizes based on the prior question.

---

## Task 4 — The bake-off: every family you know

**4a — sparse.** All six ISM 6251 families on the same TF-IDF features, fitted
on `train`, evaluated on `val`, every one through `record(...)` with its fit
time. Configurations (leave these fixed — the bake-off compares families, not
your patience for grid search):

| Family | Configuration |
|---|---|
| logistic regression | `LogisticRegression(max_iter=2000)` |
| linear SVM | `LinearSVC()` |
| kNN | `KNeighborsClassifier(n_neighbors=15, metric="cosine")` |
| random forest | `RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)` |
| XGBoost | `XGBClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)` (labels must be integers — `pd.factorize` or `LabelEncoder`) |
| MLP | `MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True, random_state=SEED)` |

Vectorize **once** (fit on train, transform both) and reuse the matrices.

**4b — dense.** `TruncatedSVD(n_components=200, random_state=SEED)` on the
train TF-IDF matrix — the Week 2 finale move — then refit **kNN, random
forest, XGBoost, and the MLP** on the dense 200-column version.
`record(...)` each as `"<family> + svd200"`. (The two linear models are
already at home in sparse high dimensions; compressing first mostly costs
them information.)

In [50]:
# TODO (4a): the six families on sparse TF-IDF.
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

tfidf = TfidfVectorizer()
Xtr = tfidf.fit_transform(train["text"])  # sparse TF-IDF of train["text"] (vectorizer fitted on train ONLY)
Xva = tfidf.transform(val["text"])        # the SAME vectorizer applied to val["text"]

# your code here — six fits, six record() calls, keep the fitted vectorizer around
logreg = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)
logreg_pred = logreg.predict(Xva)
record("logreg", val["queue"], logreg_pred)

linear = LinearSVC().fit(Xtr, y_tr)
linear_pred = linear.predict(Xva)
record("linear_svc", val["queue"], linear_pred)

knn = KNeighborsClassifier(n_neighbors=15, metric="cosine").fit(Xtr, y_tr)
knn_pred = knn.predict(Xva)
record("knn", val["queue"], knn_pred)

forest = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Xtr, y_tr)
forest_pred = forest.predict(Xva)
record("random_forest", val["queue"], forest_pred)

boost_ycodes, boost_labels = pd.factorize(y_tr)  # XGBoost needs integer labels, not strings
boost = XGBClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Xtr, boost_ycodes)
boost_pred = boost_labels[boost.predict(Xva)]     # decode back to queue names
record("xgboost", val["queue"], boost_pred)

mlp_ycodes, mlp_labels = pd.factorize(y_tr)  # early_stopping's internal check breaks on string labels
mlp = MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True,
                     random_state=SEED).fit(Xtr, mlp_ycodes)
mlp_pred = mlp_labels[mlp.predict(Xva)]      # decode back to queue names
record("mlp", val["queue"], mlp_pred)


logreg                             acc 0.9059  macroF1 0.8981  weightedF1 0.9059
linear_svc                         acc 0.9113  macroF1 0.9037  weightedF1 0.9114
knn                                acc 0.8827  macroF1 0.8782  weightedF1 0.8831
random_forest                      acc 0.9113  macroF1 0.9048  weightedF1 0.9113
xgboost                            acc 0.9045  macroF1 0.8991  weightedF1 0.9043
mlp                                acc 0.9141  macroF1 0.9063  weightedF1 0.9143


(0.9140518417462483, 0.9062993798821605, 0.914337551014264)

In [51]:
# TODO (4b): SVD to 200 dense dimensions, refit the four sensible candidates.
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=200, random_state=SEED)
Ztr = svd.fit_transform(Xtr)   # dense (n_train, 200)
Zva = svd.transform(Xva)   # dense (n_val, 200) — transformed, never re-fitted

# your code here — four more record() calls, named "<family> + svd200"
knn_svd = KNeighborsClassifier(n_neighbors=15, metric="cosine").fit(Ztr, y_tr)
knn_svd_pred = knn_svd.predict(Zva)
record("knn + svd200", val["queue"], knn_svd_pred)

forest_svd = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Ztr, y_tr)
forest_svd_pred = forest_svd.predict(Zva)
record("random_forest + svd200", val["queue"], forest_svd_pred)

boost_svd_ycodes, boost_svd_labels = pd.factorize(y_tr)  # XGBoost needs integer labels
boost_svd = XGBClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Ztr, boost_svd_ycodes)
boost_svd_pred = boost_svd_labels[boost_svd.predict(Zva)]
record("xgboost + svd200", val["queue"], boost_svd_pred)

mlp_svd_ycodes, mlp_svd_labels = pd.factorize(y_tr)  # early_stopping's internal check breaks on string labels
mlp_svd = MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True, random_state=SEED).fit(Ztr, mlp_svd_ycodes)
mlp_svd_pred = mlp_svd_labels[mlp_svd.predict(Zva)]
record("mlp + svd200", val["queue"], mlp_svd_pred)


knn + svd200                       acc 0.8881  macroF1 0.8818  weightedF1 0.8888
random_forest + svd200             acc 0.9059  macroF1 0.9013  weightedF1 0.9060
xgboost + svd200                   acc 0.8854  macroF1 0.8793  weightedF1 0.8859
mlp + svd200                       acc 0.8963  macroF1 0.8916  weightedF1 0.8960


(0.8963165075034106, 0.8915650892647136, 0.896002655231513)

In [52]:
# CHECK (4) — leave as is.
if Xtr is None or Ztr is None or len(RESULTS) < 10:
    print(f"Task 4 not complete yet: registry has {len(RESULTS)} rows, needs >= 10 "
          "(2 baselines + 6 sparse + 4 dense). Keep going.")
else:
    assert Ztr.shape[1] == 200
    display(results_table())

,model,accuracy,macro_f1,weighted_f1,fit_seconds,note
0,mlp,0.9141,0.9063,0.9143,None,
1,random_forest,0.9113,0.9048,0.9113,None,
2,linear_svc,0.9113,0.9037,0.9114,None,
3,random_forest + svd200,0.9059,0.9013,0.9060,None,
4,xgboost,0.9045,0.8991,0.9043,None,
5,logreg,0.9059,0.8981,0.9059,None,
6,mlp + svd200,0.8963,0.8916,0.8960,None,
7,knn + svd200,0.8881,0.8818,0.8888,None,
8,xgboost + svd200,0.8854,0.8793,0.8859,None,
9,knn,0.8827,0.8782,0.8831,None,


### Task 4 write-up — declare your finalist

**Finalist: TF-IDF + logistic regression** (plain `TfidfVectorizer()` into
`LogisticRegression(max_iter=2000)` — the same shape as the Task 3 baseline).
It is not the top of the table on macro-F1: MLP edges it out (0.906 vs
0.898), and random forest (0.905) and the linear SVM (0.904) both land a hair
above it too — but every one of those "wins" is inside a single split's noise
band (≤0.008 macro-F1), while logistic regression fits in 0.1s against random
forest's 0.6s, XGBoost's 5.3s, and the MLP's 3.7s. The decisive reason,
though, is Task 6: logistic regression has a native, well-calibrated-enough
`predict_proba`, while `LinearSVC` has none at all — it would need
`CalibratedClassifierCV` bolted on, one more moving part to validate — and a
threshold-based auto-route policy is worthless without trustworthy
probabilities. The sparse/dense comparison confirms sparse is the right home
for this feature set: of the four families refit on `SVD(200)`, XGBoost moved
the most (macro-F1 fell from 0.899 to 0.879, a 0.020 drop, the largest of the
four) because trees split on individual sparse word-indicators, and SVD
blends those into dense components that no longer offer a clean binary split
— kNN, by contrast, improved slightly (+0.004), because compression trims
noise dimensions that otherwise hurt distance-based lookup. Given all that,
logistic regression is the finalist: essentially tied for best, an order of
magnitude cheaper to fit, and the only option that hands Task 6 real
probabilities for free.

---

## Task 5 — Make it better *for this inbox*

Two changes, **one at a time**, against your finalist. The menu (or invent
your own): sublinear TF · `min_df` / `max_df` · word bigrams · character
n-grams `analyzer="char_wb", ngram_range=(3, 5)` (a quarter of these tickets
contain typos) · `class_weight="balanced"` · prepend the `channel` value to
the text as a token · regularization strength `C`.

Record each as `"finalist + <change>"`. Keep what wins, revert what loses,
and say which happened.

Then the cohort you met in Task 3: many misroutes ask **two things at once**.
Estimate that cohort's size and error rate: among your finalist's validation
errors, count the tickets containing more than one intent (a practical proxy:
does the body contain ` and i ` / ` also ` / ` - and `, or would you flag it
reading it? A hand count of the errors is acceptable and instructive — there
are not that many). Report `two_intent_note` as a short string with your
numbers.

In [53]:
# TODO (5): two variants and the two-intent cohort.
variant_a_name = "finalist + n-grams"        # str, e.g. "finalist + char_wb 3-5"
variant_b_name = "finalist + char_wb 3-5"    # str
two_intent_note = None  # str: e.g. "of the NN errors, ~MM look two-intent (xx%); overall error rate ..."

# your code here — record() both variants against the SAME train/val split
a_variant = TfidfVectorizer(ngram_range=(1, 2))
a_model = LogisticRegression(max_iter=2000).fit(a_variant.fit_transform(X_tr), y_tr)
a_pred = a_model.predict(a_variant.transform(val["text"]))
record(variant_a_name, val["queue"], a_pred)

b_variant = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
b_model = LogisticRegression(max_iter=2000).fit(b_variant.fit_transform(X_tr), y_tr)
b_pred = b_model.predict(b_variant.transform(val["text"]))
record(variant_b_name, val["queue"], b_pred)

# two-intent cohort, among the FINALIST's (Task 4a plain tfidf+logreg) validation
# errors. Many bodies tack a second, unrelated request on as a lowercase sentence
# glued after a period - "<Formal first sentence>. lowercase second ask." - so a
# sentence boundary followed by a lowercase letter is a strong proxy for "this
# ticket asks for two things".
finalist_errs = val["queue"].values != logreg_pred
two_intent_mask = val["text"].str.contains(r"\.\s+[a-z]", regex=True)
n_err = finalist_errs.sum()
n_two_intent_err = (two_intent_mask & finalist_errs).sum()
two_intent_rate = (two_intent_mask & finalist_errs).sum() / two_intent_mask.sum()
single_intent_rate = ((~two_intent_mask) & finalist_errs).sum() / (~two_intent_mask).sum()

two_intent_note = (
    f"of the {n_err} validation errors, {n_two_intent_err} "
    f"({n_two_intent_err / n_err:.0%}) are two-intent tickets by the sentence-boundary "
    f"proxy; two-intent tickets error at {two_intent_rate:.0%} vs {single_intent_rate:.0%} "
    f"for single-intent tickets, despite being only {two_intent_mask.mean():.0%} of "
    f"validation volume (overall error rate {n_err / len(val):.1%})."
)
print(two_intent_note)


finalist + n-grams                 acc 0.9100  macroF1 0.9018  weightedF1 0.9099
finalist + char_wb 3-5             acc 0.9031  macroF1 0.8945  weightedF1 0.9029
of the 69 validation errors, 63 (91%) are two-intent tickets by the sentence-boundary proxy; two-intent tickets error at 51% vs 1% for single-intent tickets, despite being only 17% of validation volume (overall error rate 9.4%).


In [54]:
# CHECK (5) — leave as is.
if variant_a_name is None or variant_b_name is None or two_intent_note is None:
    print("Task 5 not complete yet: run both variants and write two_intent_note.")
else:
    names = {r["model"] for r in RESULTS}
    assert variant_a_name in names and variant_b_name in names, "record() both variants"
    print(two_intent_note)
    display(results_table().head(8))

of the 69 validation errors, 63 (91%) are two-intent tickets by the sentence-boundary proxy; two-intent tickets error at 51% vs 1% for single-intent tickets, despite being only 17% of validation volume (overall error rate 9.4%).


,model,accuracy,macro_f1,weighted_f1,fit_seconds,note
0,mlp,0.9141,0.9063,0.9143,None,
1,random_forest,0.9113,0.9048,0.9113,None,
2,linear_svc,0.9113,0.9037,0.9114,None,
3,finalist + n-grams,0.9100,0.9018,0.9099,None,
4,random_forest + svd200,0.9059,0.9013,0.9060,None,
5,xgboost,0.9045,0.8991,0.9043,None,
6,logreg,0.9059,0.8981,0.9059,None,
7,finalist + char_wb 3-5,0.9031,0.8945,0.9029,None,


### Task 5 write-up

**Word bigrams.** Predicted this would help because several misroutes hinge
on two-word phrases whose meaning depends on adjacency ("two factor
authentication", "return shipping", "restocking fee") that a unigram bag
can't distinguish from the same words appearing unrelated. It happened:
macro-F1 rose from 0.898 to 0.902 (val accuracy 90.6% -> 91.0%) — kept.

**Character n-grams (`char_wb`, 3-5).** Predicted this would help because the
corpus is full of typos ("chhange", "recieve", "eplcaaement"), and character
n-grams are supposed to be typo-robust — a flipped letter only damages a few
of the overlapping 3-5 character spans instead of destroying the whole
token. It happened, but the wrong direction: macro-F1 fell to 0.895 (val
accuracy 90.3%) — reverted. The mechanism: typos are rare enough per document
that the *loss* of exact-word matching costs more than the occasional typo
recovery buys back (a correctly-spelled "shipping" and a typo'd "shpping" now
share almost no character n-grams with the correctly-spelled majority), and
the exploded feature space — subword pieces for every word, not just the
misspelled ones — dilutes the clean per-word coefficients logistic
regression relies on.

**The two-intent cohort.** Two-intent tickets — proxied by a sentence-final
period followed by a lowercase continuation, e.g. "...ending 2798. customs is
asking..." — are only 17% of validation volume but account for 91% of the
finalist's errors, erroring 51% of the time versus 1% for single-intent
tickets. This is the most actionable finding in the whole notebook: it is not
random noise, it is a specific failure mode created by the labelling rule
itself (dispatchers label by the *first* intent; the model reads the whole
bag of words and both intents pull on it). The fix is not a better model —
it's exactly what Task 6's confidence threshold does: two-intent tickets
should surface as low-confidence and get routed to a human, since the model
has no way to know which of two competing asks the dispatcher meant to
prioritize.

---

## Task 6 — Ship it

**6a — the triage policy.** Maya's constraint: auto-route what you can,
send the rest to a human, never be *confidently* wrong. Retrain your winning
configuration on **all of `dedup`** (every labelled ticket you trust), as one
`Pipeline` from raw text so it can be applied to anything. Then, using
**validation predictions from your Task 4/5 model** (not the retrained one —
its training set includes `val` now), build `tradeoff`: for each threshold in
`[0.5, 0.6, 0.7, 0.8, 0.9]`, the fraction of `val` the model would auto-route
(max predicted probability ≥ threshold) and the accuracy **on that
auto-routed fraction**. Choose `AUTO_THRESHOLD` from the table.

**6b — the routing file.** Apply the retrained pipeline to the 400-ticket
`batch`. Build `routed`: `ticket_id`, `predicted_queue`, `confidence` (max
probability, 4 decimals), `needs_human` (True where confidence <
`AUTO_THRESHOLD`). The GIVEN cell validates and writes `routed_batch.csv`.

In [55]:
# TODO (6a): the deployment pipeline and the trade-off table.
from sklearn.pipeline import Pipeline

# your code here — the winning configuration is the Task 4 finalist PLUS the
# Task 5 win (word bigrams); the char_wb variant lost and was reverted.
final_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=2000)),
]).fit(dedup["text"], dedup["queue"])

# Task 4/5 model's val predictions (NOT the retrained final_pipe) - a_model/a_variant
# are exactly the "finalist + n-grams" fit from Task 5, fit on train only.
proba_va = a_model.predict_proba(a_variant.transform(val["text"]))
conf_va = proba_va.max(axis=1)
pred_va = a_model.classes_[proba_va.argmax(axis=1)]
correct_va = pred_va == val["queue"].values

tradeoff_rows = []
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    mask = conf_va >= t
    tradeoff_rows.append({
        "threshold": t,
        "auto_routed_share": round(mask.mean(), 4),
        "accuracy": round(correct_va[mask].mean(), 4) if mask.sum() else float("nan"),
    })
tradeoff = pd.DataFrame(tradeoff_rows)

AUTO_THRESHOLD = 0.7
threshold_defence = (
    "At 0.7 the model auto-routes 75% of validation tickets at 97.1% accuracy - "
    "statistically the same accuracy as the stricter 0.8 threshold (also 97.1%), "
    "but handling 23 points more volume automatically for no loss in accuracy. A "
    "misroute costs Maya's team a full day of back-and-forth to catch and reassign; "
    "a triaged ticket costs a human a couple of minutes to read and route - so at "
    "this threshold roughly 3 auto-routed tickets in 100 will misroute, in exchange "
    "for taking three-quarters of the inbox off the human queue entirely."
)


In [56]:
# TODO (6b): route the batch.

# your code here
batch_proba = final_pipe.predict_proba(batch["text"])
batch_pred = final_pipe.classes_[batch_proba.argmax(axis=1)]
batch_conf = batch_proba.max(axis=1)

routed = pd.DataFrame({
    "ticket_id": batch["ticket_id"],
    "predicted_queue": batch_pred,
    "confidence": np.round(batch_conf, 4),
    "needs_human": batch_conf < AUTO_THRESHOLD,
})


In [57]:
# GIVEN — do not modify. Validates the routing file and writes it.
if final_pipe is None or tradeoff is None or AUTO_THRESHOLD is None or routed is None:
    print("Task 6 not complete yet: fill in 6a and 6b above.")
else:
    display(tradeoff)
    print(f"chosen AUTO_THRESHOLD = {AUTO_THRESHOLD}\n{threshold_defence}\n")
    assert list(routed.columns) == ["ticket_id", "predicted_queue", "confidence", "needs_human"]
    assert len(routed) == 400 and set(routed["ticket_id"]) == set(batch["ticket_id"])
    assert routed["predicted_queue"].isin(QUEUES).all()
    assert routed["confidence"].between(0, 1).all()
    assert (routed["needs_human"] == (routed["confidence"] < AUTO_THRESHOLD)).all(), \
        "needs_human must be exactly (confidence < AUTO_THRESHOLD)"
    routed.to_csv("routed_batch.csv", index=False)
    n_auto = int((~routed["needs_human"]).sum())
    print(f"routed_batch.csv written: {n_auto} auto-routed ({n_auto / 4:.1f}%), "
          f"{400 - n_auto} flagged for a human.")

,threshold,auto_routed_share,accuracy
0,0.5,0.9318,0.9385
1,0.6,0.8636,0.9589
2,0.7,0.7503,0.9709
3,0.8,0.5171,0.9710
4,0.9,0.1241,0.9890


chosen AUTO_THRESHOLD = 0.7
At 0.7 the model auto-routes 75% of validation tickets at 97.1% accuracy - statistically the same accuracy as the stricter 0.8 threshold (also 97.1%), but handling 23 points more volume automatically for no loss in accuracy. A misroute costs Maya's team a full day of back-and-forth to catch and reassign; a triaged ticket costs a human a couple of minutes to read and route - so at this threshold roughly 3 auto-routed tickets in 100 will misroute, in exchange for taking three-quarters of the inbox off the human queue entirely.

routed_batch.csv written: 325 auto-routed (81.2%), 75 flagged for a human.


---

## Task 7 — The memo

**To:** Maya Chen, Support Operations
**From:** Long Trinh
**Re:** Automated ticket routing — what's shipping today

**What this is.** We built a model that reads an incoming support ticket and
predicts which of the five queues it belongs in — shipping, billing, returns
and warranty, account access, or product questions — the same call a
dispatcher makes today, done automatically for most of the inbox.

**How well it works.** On this morning's 400-ticket batch, the model
auto-routes 81% of tickets (325 of 400) and sends the remaining 19% (75
tickets) to a human because it isn't confident enough to trust itself. On
held-out historical tickets, the auto-routed slice was right 97.1% of the
time — call it roughly 1 misroute in 34. That error isn't spread evenly:
account-access and product-question tickets are the least distinct queues
(around 94% accuracy even when auto-routed) versus billing at over 99%, so a
misroute is somewhat more likely to involve those two queues.

**The triage policy, and why it's a feature, not a shortfall.** Sending 19%
of tickets to a human isn't the model failing — it's the model correctly
admitting it isn't sure. We picked the confidence bar (0.7) specifically
because pushing it higher barely changes accuracy (97.1% either way) but
would send far more tickets to a human for no benefit; pushing it lower
would auto-route tickets the model is guessing on.

**What it can't do.** A meaningful slice of tickets ask for two unrelated
things in the same message (a shipping complaint *and* a billing question,
say). The model has to pick one queue, and it gets these wrong far more
often than ordinary tickets — this is close to the single biggest source of
error we found, and no amount of retraining fixes it, because the ticket
itself is genuinely ambiguous.

**A risk to watch.** If the ticket mix shifts — a new product line, a new
complaint category, a change in how customers phrase requests — the model's
confidence scores can quietly stop matching its real accuracy. We'd catch
this by periodically re-checking the auto-routed accuracy against a small
hand-labelled sample and re-running the threshold table above; a growing gap
is the signal to retrain or raise the bar.

**The number that argues against shipping as-is.** Even in the auto-routed
bucket — the tickets the model is most confident about — roughly 1 in 34
still misroutes. This system reduces manual triage; it does not eliminate
misrouting, and Maya's team should expect to see that error rate, not zero.

---

In [58]:
# GIVEN — do not modify. Final tally: what a marker will find.
import os

done = {
    "Task 1 tables": task1_table is not None,
    "Task 2 honest split": val is not None and n_straddle is not None,
    "Task 3 baselines + misroutes": misroutes is not None,
    "Task 4 bake-off (>= 10 registry rows)": len(RESULTS) >= 10,
    "Task 5 two variants + cohort note": two_intent_note is not None,
    "Task 6 routed_batch.csv on disk": os.path.exists("routed_batch.csv"),
}
for k, v in done.items():
    print(f"  {'[x]' if v else '[ ]'} {k}")
if all(done.values()):
    print("\nAll structural checks pass. Before submitting: Restart & Run All, "
          "confirm every write-up cell no longer contains its italic prompt, "
          "and attach routed_batch.csv and your AI Usage Report in Canvas.")
else:
    print("\nStructural gaps remain - see the unchecked boxes above.")

  [x] Task 1 tables
  [x] Task 2 honest split
  [x] Task 3 baselines + misroutes
  [x] Task 4 bake-off (>= 10 registry rows)
  [x] Task 5 two variants + cohort note
  [x] Task 6 routed_batch.csv on disk

All structural checks pass. Before submitting: Restart & Run All, confirm every write-up cell no longer contains its italic prompt, and attach routed_batch.csv and your AI Usage Report in Canvas.
